In [1]:
import requests
import pandas as pd
import os
import matplotlib.pyplot as plt

In [ ]:
URL = "https://api.energidataservice.dk/dataset/PrivateConsumptionHeatingHour"

params = {
    "start": "2025-01-01",
    "end": "2025-12-31",
    "filter": '{"MunicipalityCode":["461"]}',
    "sort": "TimeDK asc",
    "limit": 0
}

In [ ]:
parquet_file_path = r"C:\Users\simon\Desktop\Data Engineering Journey\Weather-Energy_project\odense_heating_consumption_2025.parquet"

if os.path.exists(parquet_file_path):
    
    print("Parquet file found. Loading local data")
    df = pd.read_parquet(parquet_file_path)

else:    
    
    response = requests.get(URL, params=params, timeout=5) # The response Object contains the server's response to the HTTP request.
    
    # Raise an HTTPError object, if an error occur.
    response.raise_for_status() 
    
    if response.status_code == 200:
        
        # Retrieve the data from the response 
        data = response.json()
        print("Download completed!")
        
        # Transform the JSON dataset to a dataframe
        df = pd.DataFrame(data["records"])

        # Save the data in parquet format: 
        df.to_parquet("odense_heating_consumption_2025.parquet", index=False)


A **Parquet** is a columnar storage file format for which data are stored column by column instead of row by row (like CSV and JSON).
It is efficient for data storage and analytical queries

In [ ]:
print(f"Total records:{len(df)}\n")
print(df.info())

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print(df.columns.tolist())

In [ ]:
print("The different housing categories:\n",df["HousingCategory"].unique())
print("\nThe different heating categories:\n",df["HeatingCategory"].unique())

In [ ]:
print("number of records:",len(df["TimeDK"]))
print("Count of the total number of days in the dataset",(len(df["TimeDK"])//2)//24)

In [ ]:
if 'timeUTC' in df.columns:
    df = df.drop('TimeUTC', axis=1)
else:
    print("timeUTC column already dropped")

# Convert the Municipality code into a string
df['MunicipalityCode'] = df['MunicipalityCode'].astype(str)


In [ ]:
print("Electricity Consumption Summary")
df.describe()

In [ ]:
# Group per heating category:
print("Electricity Consumption Summary per Heating category")
df.groupby("HeatingCategory").describe()

In [ ]:
df["TimeDK"] = pd.to_datetime(df["TimeDK"])
df["Date"] = df["TimeDK"].dt.date

In [ ]:
daily_consumption_df = (
    df.groupby(["Date", "HeatingCategory"])["ConsumptionkWh"]
    .sum()
    .reset_index()
)

In [ ]:
daily_consumption_df.head()

In [ ]:
plt.figure(figsize=(15,6))


for category in daily_consumption_df["HeatingCategory"].unique():

    category_df = daily_consumption_df[
        daily_consumption_df["HeatingCategory"] == category
    ]

    plt.plot(
        category_df["Date"],
        category_df["ConsumptionkWh"],
        label=category
    )

    plt.title("Daily Heating Consumption in Odense in 2025")
    plt.xlabel("Date")
    plt.ylabel("Consumption in kWh")
    plt.legend(title = "Heating Category")
    plt.grid(alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.show

In [ ]:
daily_consumption_df["DailyShare"] = (
    daily_consumption_df
    .groupby("HeatingCategory")["ConsumptionkWh"]
    .transform(lambda x: x / x.sum() * 100)
)

In [ ]:
plt.figure(figsize=(15, 6))

for category in daily_consumption_df["HeatingCategory"].unique():

    category_data = daily_consumption_df[
        daily_consumption_df["HeatingCategory"] == category
    ]

    plt.plot(
        category_data["Date"],
        category_data["DailyShare"],
        label=category
    )

plt.title("Daily Share of Annual Consumption by Heating Category")
plt.xlabel("Date")
plt.ylabel("Share of Annual Consumption (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
daily_consumption_df["Month"] = pd.to_datetime(
    daily_consumption_df["Date"]
).dt.month

monthly_consumption_df = (
    daily_consumption_df
    .groupby(["Month", "HeatingCategory"])["ConsumptionkWh"]
    .sum()
    .reset_index()
)

monthly_consumption_df["MonthlyShare"] = (
    monthly_consumption_df
    .groupby("HeatingCategory")["ConsumptionkWh"]
    .transform(lambda x: x / x.sum() * 100)
)

In [ ]:
plt.figure(figsize=(15, 6))

for category in monthly_consumption_df["HeatingCategory"].unique():

    category_data = monthly_consumption_df[
        monthly_consumption_df["HeatingCategory"] == category
    ]

    plt.plot(
        category_data["Month"],
        category_data["MonthlyShare"],
        label=category
    )

plt.title("Daily Share of Annual Consumption by Heating Category")
plt.xlabel("Month")
plt.ylabel("Share of Annual Consumption (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
3/5 * (4**4/5**4)